# H-bond Partner Distribution: Real vs Imputed Waters

Compare the residue-type and atom-role distribution of H-bond partners for:
- **Real waters** with ≥ 3 H-bond partners (from the GT structure)
- **Imputed waters** placed geometrically from the stripped structure

For each partner atom we record:
| Field | Values |
|---|---|
|  | PROTEIN / DNA / RNA / NONPOLYMER / SOLVENT |
|  | three-letter residue code |
|  | e.g. N, OG, NE2 |
|  | backbone / sidechain / base / N/A |
|  | donor / acceptor / both / unknown |

All helper functions live in .

## Setup

In [1]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path(".").resolve()))

from boltzgen.data import const
from hbond_partner_analysis import (
    get_mol_category, classify_backbone_vs_sidechain, get_hbond_role,
    build_atom_to_residue_map, build_atom_to_chain_map, get_hbond_candidate_info,
    find_hbond_partners, describe_partner, get_water_partner_descriptors,
    flatten_descriptors,
    load_gt_structure, filter_to_min_hbonds, build_imputed_structure,
    extract_solvent_coords,
    analyze_one_pdb,
    collect_analysis_results,
    get_flat_descriptors,
    get_partner_counts,
    get_all_partner_counts,
    plot_distribution_comparison,
    plot_hbond_count_histogram,
    plot_all_distributions,
    HBOND_MIN_DIST, HBOND_MAX_DIST, NPZ_ROOT,
)

PDB_IDS = Path("easy.txt").read_text().split()
print(f"{len(PDB_IDS)} structures in easy.txt")

43 structures in easy.txt


---
## Part 1: single-structure walkthrough

Walk through the analysis for one PDB entry step by step so every intermediate is inspectable.

### 1a. Load the GT structure

In [ ]:
DEMO_PDB = PDB_IDS[0]
print(f"Using {DEMO_PDB}")

gt = load_gt_structure(DEMO_PDB)
print(f"Total chains: {len(gt.chains)}")
print(f"Total atoms:  {len(gt.atoms)}")
mol_type_counts = {}
for chain in gt.chains:
    cat = get_mol_category(chain["mol_type"])
    mol_type_counts[cat] = mol_type_counts.get(cat, 0) + 1
print(f"Chains by type: {mol_type_counts}")

### 1b. Build H-bond candidate info — from stripped (non-water) structure

We use only non-water atoms as potential H-bond partners, keeping the analysis
symmetric between real and imputed waters. Solvent-solvent contacts are excluded.

In [ ]:
cand = get_hbond_candidate_info(gt.remove_solvents())
print(f"Total non-water H-bond candidate atoms: {len(cand['atom_indices'])}")
cats, cnts = np.unique(cand["mol_categories"], return_counts=True)
for c, n in zip(cats, cnts):
    print(f"  {c}: {n}")

### 1c. Extract real ≥3-hbond water coordinates

In [ ]:
MIN_HBONDS = 3
real_gt = filter_to_min_hbonds(gt, min_hbonds=MIN_HBONDS)
real_coords = extract_solvent_coords(real_gt, mol_types=(const.chain_type_ids["SOLVENT"],))
print(f"Real waters with >= {MIN_HBONDS} H-bonds: {len(real_coords)}")

### 1d. Find H-bond partners for each real water

In [ ]:
real_partners = find_hbond_partners(real_coords, cand,
                                    min_dist=HBOND_MIN_DIST, max_dist=HBOND_MAX_DIST)
partner_counts = [len(p) for p in real_partners]
print(f"Partner counts — min: {min(partner_counts)}, mean: {np.mean(partner_counts):.1f}, max: {max(partner_counts)}")

### 1e. Convert partner atom indices to descriptor dicts

In [ ]:
real_descriptors = [
    [describe_partner(cand, idx) for idx in partners]
    for partners in real_partners
]

# Show a few example descriptors for the first water
print(f"Partners for water 0:")
for d in real_descriptors[0]:
    print(f"  {d}")

---
### 1f. Build the imputed structure

Takes the GT structure, strips all waters, imputes geometrically, and filters clashes.
The returned Structure has  chains for surviving imputed waters.

In [ ]:
MAX_HBOND_LENGTH   = 3.5
PROTEIN_CLASH_RAD  = 2.0
SOLVENT_CLASH_RAD  = 2.0

imputed = build_imputed_structure(
    gt,
    max_hbond_length=MAX_HBOND_LENGTH,
    protein_clash_rad=PROTEIN_CLASH_RAD,
    solvent_clash_rad=SOLVENT_CLASH_RAD,
)

### 1g. Extract imputed water coordinates from the imputed structure

In [ ]:
imputed_coords = extract_solvent_coords(imputed, mol_types=(const.chain_type_ids["iSOLVENT"],))
print(f"Surviving imputed waters: {len(imputed_coords)}")

### 1h. Build candidate info for partner lookup — from stripped structure

Imputed waters were placed using *only* non-water atoms, so we query partners
from the stripped (water-free) structure. This avoids counting spurious
imputed-water-to-imputed-water contacts as H-bonds.

In [ ]:
stripped_cand = get_hbond_candidate_info(gt.remove_solvents())
print(f"Non-water H-bond candidate atoms: {len(stripped_cand['atom_indices'])}")

### 1i. Find and describe H-bond partners for imputed waters

In [ ]:
imputed_partners = find_hbond_partners(imputed_coords, stripped_cand,
                                       min_dist=HBOND_MIN_DIST, max_dist=HBOND_MAX_DIST)
imputed_descriptors = [
    [describe_partner(stripped_cand, idx) for idx in partners]
    for partners in imputed_partners
]
imp_counts = [len(p) for p in imputed_partners]
print(f"Imputed partner counts — min: {min(imp_counts)}, mean: {np.mean(imp_counts):.1f}, max: {max(imp_counts)}")

---
## Part 2: run across all easy.txt structures

 calls  for each entry and returns
flat lists of all partner dicts across all waters across all structures.

In [ ]:
all_results = collect_analysis_results(
    PDB_IDS,
    min_hbonds=MIN_HBONDS,
    max_hbond_length=MAX_HBOND_LENGTH,
    protein_clash_rad=PROTEIN_CLASH_RAD,
    solvent_clash_rad=SOLVENT_CLASH_RAD,
    verbose=True,
)

# Derive flat descriptors and per-water counts in one pass.
real_flat, imputed_flat = get_flat_descriptors(all_results)
real_counts, imputed_counts = get_all_partner_counts(all_results)

print(f"Total partner contacts: real={len(real_flat)}, imputed={len(imputed_flat)}")
print(f"Total waters:           real={len(real_counts)}, imputed={len(imputed_counts)}")

### H-bond partner count per water

One histogram bar per integer count. Real waters are left-truncated at 3
(by the min_hbonds filter). Imputed waters were placed for exactly 3 atom
partners but the strict [min, max] distance window may yield different counts.

In [ ]:
fig = plot_hbond_count_histogram(real_counts, imputed_counts)
plt.show()

---
## Part 3: compare distributions

Each plot is a normalized bar chart (fraction of all partner contacts) so the two
populations are directly comparable despite different absolute counts.

### 3a. Molecule category

In [ ]:
fig = plot_distribution_comparison(
    [d["mol_category"] for d in real_flat],
    [d["mol_category"] for d in imputed_flat],
    title="H-bond partner molecule type",
    xlabel="mol_category",
)
plt.show()

### 3b. Backbone vs sidechain vs base

In [ ]:
fig = plot_distribution_comparison(
    [d["bb_sc"] for d in real_flat],
    [d["bb_sc"] for d in imputed_flat],
    title="H-bond partner backbone vs sidechain",
    xlabel="bb_sc",
)
plt.show()

### 3c. H-bond role (donor / acceptor / both / unknown)

In [ ]:
fig = plot_distribution_comparison(
    [d["role"] for d in real_flat],
    [d["role"] for d in imputed_flat],
    title="H-bond partner chemical role",
    xlabel="role",
)
plt.show()

### 3d. Residue type (top 20)

In [ ]:
fig = plot_distribution_comparison(
    [d["res_name"] for d in real_flat],
    [d["res_name"] for d in imputed_flat],
    title="H-bond partner residue type (top 20)",
    xlabel="res_name",
    top_n=20,
)
plt.show()

### 3e. Residue type — protein only, backbone vs sidechain split

Restrict to PROTEIN partners and split by bb_sc to see if real/imputed differ
in how much they prefer backbone vs sidechain contacts, and which residues dominate each.

In [ ]:
for bb_sc_val in ["backbone", "sidechain"]:
    real_sub    = [d["res_name"] for d in real_flat    if d["mol_category"] == "PROTEIN" and d["bb_sc"] == bb_sc_val]
    imputed_sub = [d["res_name"] for d in imputed_flat if d["mol_category"] == "PROTEIN" and d["bb_sc"] == bb_sc_val]
    if not real_sub and not imputed_sub:
        continue
    fig = plot_distribution_comparison(
        real_sub, imputed_sub,
        title=f"Protein {bb_sc_val} residue type (top 20)",
        xlabel="res_name",
        top_n=20,
    )
    plt.show()